In [13]:
import numpy as np
import tensorflow as tf 
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,Input
import warnings 
warnings.filterwarnings('ignore')

In [14]:
with open("datasets/next_word_prediction.txt", "r", encoding="cp1252", errors="replace") as f:
    data = f.read()

In [7]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

NameError: name 'data' is not defined

In [4]:
len(tokenizer.word_index)

8931

In [5]:
input_seq=[]
for sentence in data.split('\n'):
    tokenized_sentence= tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1,len(tokenized_sentence)):
        input_seq.append(tokenized_sentence[:i+1])

In [6]:
max_len = max([len(x) for x in input_seq])
print(max_len)

20


## Padding

In [7]:
padded_input_seq = pad_sequences(input_seq , max_len , padding = 'pre')

X = padded_input_seq[:,:-1]
y = padded_input_seq[:,-1]

y = to_categorical(y,num_classes = 8932)
y.shape

(101619, 8932)

## LSTM Model

In [8]:
model = Sequential([
    Input(shape=(20,)),
    Embedding(input_dim=8932, output_dim=400),
    LSTM(100),
    Dense(8932,activation='softmax')
])

In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 20, 400)             │       3,572,800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 100)                 │         200,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 8932)                │         902,132 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,675,332 (17.83 MB)

 Trainable params: 4,675,332 (17.83 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.compile(loss = "categorical_crossentropy" , optimizer ='adam' , metrics = ['accuracy'])

In [11]:
Model: "sequential"

In [ ]:
model.fit(X,y,epochs = 25)

Epoch 1/25
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 62s 19ms/step - accuracy: 0.0824 - loss: 6.2165
Epoch 2/25
2710/3176 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - accuracy: 0.1311 - loss: 5.4597

In [ ]:
sample_text = "precisely"
for i in range(4):
    tokenized_text = tokenizer.texts_to_sequences([sample_text])[0]
    
    padded_text = pad_sequences([tokenized_text],maxlen = 20, padding= 'pre')
    
    nxt_word = np.argmax(model.predict(padded_text))
    
    for word, index in tokenizer.word_index.items():
        if index == nxt_word:
            sample_text = sample_text+" "+word
print(sample_text)